# 🛒 RPC Dataset — Exploratory Data Analysis

**Retail Product Checkout (RPC)** is a large-scale benchmark dataset for **checkout-free retail**.
It contains **~83,739 product instances** spanning **200 SKU categories** across 17 super-categories.

| Split | Role | Notes |
|-------|------|-------|
| `train2019` | Single-product images | Clean, isolated, one product per image |
| `val2019`   | Checkout scenes | Multiple products, 3 clutter levels |
| `test2019`  | Checkout scenes | Multiple products, 3 clutter levels |

This notebook covers:
1. Environment setup & dataset download
2. JSON structure inspection
3. Category & super-category analysis
4. Split-level statistics (images, objects, density)
5. Image dimension analysis
6. Clutter level distribution
7. Bounding box analysis (size, aspect ratio)
8. Class distribution & imbalance
9. Annotated sample visualization
10. Objects-per-image analysis

---
## 1. Environment Setup

In [ ]:
# ── Install / upgrade packages ──────────────────────────────────────────────
!pip install -q ultralytics opencv-python-headless matplotlib seaborn pandas numpy Pillow tqdm

import os, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter
import cv2

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.precision', 3)

# ── GPU check ───────────────────────────────────────────────────────────────
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU avail: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device   : {torch.cuda.get_device_name(0)}')

---
## 2. Dataset Download

> The RPC dataset is hosted on Kaggle. The cell below downloads it via the Kaggle API.
> Upload your `kaggle.json` credentials or set `KAGGLE_USERNAME` / `KAGGLE_KEY` env vars first.

In [ ]:
import os

# ── Option A: Kaggle API ────────────────────────────────────────────────────
# from google.colab import files
# files.upload()   # upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d diyer22/retail-product-checkout-dataset --unzip -p /content/rpc

# ── Option B: Google Drive (if already downloaded) ──────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_ROOT = '/content/drive/MyDrive/rpc'

# ── Set DATA_ROOT ──────────────────────────────────────────────────────────
DATA_ROOT = '/content/rpc'   # ← change if needed

# ── Verify structure ──────────────────────────────────────────────────────
expected = ['instances_train2019.json', 'instances_val2019.json',
            'instances_test2019.json', 'train2019', 'val2019', 'test2019']
for item in expected:
    status = '✅' if os.path.exists(os.path.join(DATA_ROOT, item)) else '❌'
    print(f'{status}  {item}')

---
## 3. Load Annotations

In [ ]:
def load_json(path):
    with open(path, 'rb') as f:
        return json.load(f)

train_js = load_json(os.path.join(DATA_ROOT, 'instances_train2019.json'))
val_js   = load_json(os.path.join(DATA_ROOT, 'instances_val2019.json'))
test_js  = load_json(os.path.join(DATA_ROOT, 'instances_test2019.json'))

print('Keys in train JSON:', list(train_js.keys()))
print('Keys in val   JSON:', list(val_js.keys()))
print('Keys in test  JSON:', list(test_js.keys()))

In [ ]:
# Dataset metadata
from pprint import pprint
print('=== info ===')
pprint(train_js.get('info', {}))
print('\n=== sample image record ===')
pprint(val_js['images'][0])
print('\n=== sample annotation record ===')
pprint(val_js['annotations'][0])

---
## 4. Category Analysis

RPC has **200 SKU-level categories** grouped under **17 super-categories**.

In [ ]:
cat_df = pd.DataFrame(train_js['categories'])
print(f'Total categories : {len(cat_df)}')
print(f'Super-categories : {cat_df["supercategory"].nunique()}')
cat_df.head(10)

In [ ]:
# SKUs per super-category
skus_per_super = cat_df.groupby('supercategory').size().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(skus_per_super.index, skus_per_super.values,
              color=sns.color_palette('muted', len(skus_per_super)))
ax.bar_label(bars, padding=2, fontsize=9)
ax.set_title('Number of SKU Categories per Super-Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Super-Category')
ax.set_ylabel('Number of SKUs')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(skus_per_super.to_string())

In [ ]:
# Chinese SKU name lookup (available in val/test)
raw_df = pd.DataFrame(val_js.get('__raw_Chinese_name_df', []))
print(f'Raw name records: {len(raw_df)}')
raw_df.head(10)

In [ ]:
# SKU class distribution (17 product classes)
if len(raw_df):
    sku_class_counts = raw_df['sku_class'].value_counts()
    fig, ax = plt.subplots(figsize=(12, 4))
    sns.barplot(x=sku_class_counts.index, y=sku_class_counts.values, ax=ax,
                palette='Set2')
    ax.bar_label(ax.containers[0], padding=2, fontsize=8)
    ax.set_title('SKUs per Product Class (sku_class)', fontsize=13, fontweight='bold')
    ax.set_xlabel('sku_class')
    ax.set_ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

---
## 5. Split-Level Statistics

In [ ]:
def split_statistics(js, split_name):
    ann_df = pd.DataFrame(js['annotations'])
    n_images = len(js['images'])
    n_objects = len(ann_df)
    objs_per_img  = ann_df.groupby('image_id')['id'].count().mean()
    cats_per_img  = ann_df.groupby('image_id')['category_id'].nunique().mean()
    return {
        'split'             : split_name,
        'images'            : n_images,
        'annotations'       : n_objects,
        'obj/image (mean)'  : round(objs_per_img, 2),
        'cats/image (mean)' : round(cats_per_img, 2),
    }

checkout_js = {
    'images'      : val_js['images'] + test_js['images'],
    'annotations' : val_js['annotations'] + test_js['annotations'],
}

stats = pd.DataFrame([
    split_statistics(train_js, 'train'),
    split_statistics(val_js,   'val'),
    split_statistics(test_js,  'test'),
    split_statistics(checkout_js, 'checkout (val+test)'),
])
stats

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
splits3 = stats[stats['split'].isin(['train', 'val', 'test'])].copy()

for ax, col, title in zip(axes,
    ['images', 'annotations', 'obj/image (mean)'],
    ['Images per Split', 'Annotations per Split', 'Mean Objects per Image']):
    bars = ax.bar(splits3['split'], splits3[col],
                  color=['#4C72B0', '#DD8452', '#55A868'])
    ax.bar_label(bars, fmt='%.1f', padding=3)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(col)

plt.suptitle('RPC Dataset — Split Summary', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Image Dimension Analysis

In [ ]:
for split_name, js in [('train', train_js), ('val', val_js), ('test', test_js)]:
    img_df = pd.DataFrame(js['images'])
    print(f'\n── {split_name.upper()} ─────────────────────────────')
    print(img_df[['width', 'height']].describe().T.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (split_name, js) in zip(axes,
    [('train', train_js), ('val', val_js), ('test', test_js)]):
    img_df = pd.DataFrame(js['images'])
    ax.scatter(img_df['width'], img_df['height'], alpha=0.3, s=10)
    ax.set_title(f'{split_name} — W×H scatter', fontweight='bold')
    ax.set_xlabel('Width (px)')
    ax.set_ylabel('Height (px)')
    # Annotate unique sizes
    size_counts = img_df.groupby(['width','height']).size().reset_index(name='n')
    for _, r in size_counts.iterrows():
        ax.annotate(f"{int(r['width'])}×{int(r['height'])} (n={r['n']})",
                    xy=(r['width'], r['height']), fontsize=7,
                    xytext=(5, 5), textcoords='offset points')

plt.suptitle('Image Dimensions per Split', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Clutter Level Distribution

Checkout images (val + test) are annotated with a `level` field: `easy`, `medium`, or `hard`.
This reflects how many products are mixed together in the checkout scene.

In [ ]:
# Level statistics for val + test
level_rows = []
for split_name, js in [('val', val_js), ('test', test_js)]:
    img_df  = pd.DataFrame(js['images'])
    ann_df  = pd.DataFrame(js['annotations'])
    for level in ['easy', 'medium', 'hard']:
        lvl_imgs   = img_df[img_df['level'] == level]
        lvl_ids    = set(lvl_imgs['id'])
        lvl_anns   = ann_df[ann_df['image_id'].isin(lvl_ids)]
        objs_mean  = lvl_anns.groupby('image_id')['id'].count().mean() if len(lvl_anns) else 0
        level_rows.append({
            'split'  : split_name,
            'level'  : level,
            'images' : len(lvl_imgs),
            'anns'   : len(lvl_anns),
            'obj/img': round(objs_mean, 2),
        })

lvl_df = pd.DataFrame(level_rows)
print(lvl_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
level_order = ['easy', 'medium', 'hard']
palette     = {'easy': '#55A868', 'medium': '#DD8452', 'hard': '#C44E52'}

for ax, col, title in zip(axes, ['images', 'obj/img'],
                           ['Images per Level & Split', 'Mean Objects/Image per Level & Split']):
    sns.barplot(data=lvl_df, x='level', y=col, hue='split',
                order=level_order, ax=ax)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Clutter Level')

# Pie charts for level proportions (combined val+test)
combined = lvl_df.groupby('level')['images'].sum()
fig2, ax2 = plt.subplots(figsize=(5, 5))
wedge_colors = [palette[l] for l in level_order]
ax2.pie([combined.get(l, 0) for l in level_order],
        labels=level_order, autopct='%.1f%%',
        colors=wedge_colors, startangle=140)
ax2.set_title('Checkout Images — Clutter Level Split', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 8. Bounding Box Analysis

In [ ]:
# Build a unified annotation DataFrame across all splits
dfs = []
for split_name, js in [('train', train_js), ('val', val_js), ('test', test_js)]:
    df = pd.DataFrame(js['annotations'])
    df['split'] = split_name
    img_df = pd.DataFrame(js['images'])[['id', 'width', 'height']]
    img_df.rename(columns={'id': 'image_id', 'width': 'img_w', 'height': 'img_h'}, inplace=True)
    df = df.merge(img_df, on='image_id', how='left')
    dfs.append(df)

all_ann = pd.concat(dfs, ignore_index=True)

# Extract bbox components
all_ann[['bx', 'by', 'bw', 'bh']] = pd.DataFrame(all_ann['bbox'].tolist(), index=all_ann.index)
all_ann['bbox_area']    = all_ann['bw'] * all_ann['bh']
all_ann['aspect_ratio'] = all_ann['bw'] / all_ann['bh'].replace(0, np.nan)

# Normalised (relative to image size)
all_ann['bw_rel'] = all_ann['bw'] / all_ann['img_w']
all_ann['bh_rel'] = all_ann['bh'] / all_ann['img_h']
all_ann['area_rel'] = all_ann['bw_rel'] * all_ann['bh_rel']

print(all_ann[['bw', 'bh', 'bbox_area', 'aspect_ratio', 'bw_rel', 'bh_rel']].describe())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. BBox width distribution
for split in ['train', 'val', 'test']:
    axes[0, 0].hist(all_ann[all_ann['split']==split]['bw'], bins=60, alpha=0.5, label=split)
axes[0, 0].set_title('BBox Width Distribution')
axes[0, 0].set_xlabel('Width (px)')
axes[0, 0].legend()

# 2. BBox height distribution
for split in ['train', 'val', 'test']:
    axes[0, 1].hist(all_ann[all_ann['split']==split]['bh'], bins=60, alpha=0.5, label=split)
axes[0, 1].set_title('BBox Height Distribution')
axes[0, 1].set_xlabel('Height (px)')
axes[0, 1].legend()

# 3. BBox area distribution (log scale)
for split in ['train', 'val', 'test']:
    axes[0, 2].hist(np.log1p(all_ann[all_ann['split']==split]['bbox_area']), bins=60, alpha=0.5, label=split)
axes[0, 2].set_title('BBox Area Distribution (log scale)')
axes[0, 2].set_xlabel('log(1 + area)')
axes[0, 2].legend()

# 4. Aspect ratio distribution
clip = all_ann['aspect_ratio'].clip(0, 4)
axes[1, 0].hist(clip, bins=80, color='steelblue', edgecolor='none', alpha=0.7)
axes[1, 0].axvline(1.0, color='red', linestyle='--', label='Square')
axes[1, 0].set_title('Aspect Ratio Distribution (all splits)')
axes[1, 0].set_xlabel('Width / Height')
axes[1, 0].legend()

# 5. Relative BBox size scatter
sample = all_ann.sample(min(5000, len(all_ann)), random_state=42)
axes[1, 1].scatter(sample['bw_rel'], sample['bh_rel'], alpha=0.2, s=8)
axes[1, 1].set_title('Relative BBox Size (W/img_W vs H/img_H)')
axes[1, 1].set_xlabel('Relative Width')
axes[1, 1].set_ylabel('Relative Height')

# 6. BBox area CDF
sorted_area = np.sort(all_ann['area_rel'].dropna())
cdf = np.arange(1, len(sorted_area)+1) / len(sorted_area)
axes[1, 2].plot(sorted_area, cdf)
axes[1, 2].set_title('CDF of Relative BBox Area')
axes[1, 2].set_xlabel('Relative Area')
axes[1, 2].set_ylabel('CDF')
for pct, color in [(0.5, 'green'), (0.75, 'orange'), (0.95, 'red')]:
    val = np.percentile(sorted_area, pct*100)
    axes[1, 2].axvline(val, color=color, linestyle='--', label=f'P{int(pct*100)}={val:.3f}')
axes[1, 2].legend(fontsize=8)

plt.suptitle('Bounding Box Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Class Distribution & Imbalance

In [ ]:
# Build category id -> name lookup
id2name = {c['id']: c['name'] for c in train_js['categories']}
id2super = {c['id']: c['supercategory'] for c in train_js['categories']}

# Annotation counts per category for TRAIN
train_ann = all_ann[all_ann['split'] == 'train'].copy()
cat_counts = train_ann['category_id'].value_counts().reset_index()
cat_counts.columns = ['category_id', 'count']
cat_counts['name']  = cat_counts['category_id'].map(id2name)
cat_counts['super'] = cat_counts['category_id'].map(id2super)

print(f'Most common: {cat_counts.iloc[0]["name"]}  ({cat_counts.iloc[0]["count"]:,} instances)')
print(f'Least common: {cat_counts.iloc[-1]["name"]}  ({cat_counts.iloc[-1]["count"]:,} instances)')
print(f'Imbalance ratio (max/min): {cat_counts["count"].max() / cat_counts["count"].min():.1f}x')
cat_counts.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Top 30 categories
top30 = cat_counts.head(30)
colors30 = [sns.color_palette('tab20')[i % 20] for i in range(30)]
axes[0].barh(top30['name'][::-1], top30['count'][::-1], color=colors30[::-1])
axes[0].set_title('Top 30 Categories by Instance Count (Train)', fontweight='bold')
axes[0].set_xlabel('Instances')

# Log-scale distribution of ALL 200 categories
axes[1].plot(range(len(cat_counts)), cat_counts['count'].values, marker='o', markersize=3, linewidth=1)
axes[1].set_yscale('log')
axes[1].set_title('Instance Count per Category — Long Tail (log scale)', fontweight='bold')
axes[1].set_xlabel('Category rank')
axes[1].set_ylabel('Instances (log)')
axes[1].fill_between(range(len(cat_counts)), cat_counts['count'].values, alpha=0.15)

plt.tight_layout()
plt.show()

In [ ]:
# Instances per super-category, split
all_ann['supercategory'] = all_ann['category_id'].map(id2super)
pivot = all_ann.groupby(['supercategory', 'split']).size().unstack(fill_value=0)
pivot = pivot[['train', 'val', 'test']]

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.3, ax=ax)
ax.set_title('Instance Count per Super-Category × Split', fontsize=13, fontweight='bold')
ax.set_xlabel('Split')
ax.set_ylabel('Super-Category')
plt.tight_layout()
plt.show()

---
## 10. Objects-per-Image Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (split_name, js) in zip(axes,
    [('train', train_js), ('val', val_js), ('test', test_js)]):
    ann_df = pd.DataFrame(js['annotations'])
    opc    = ann_df.groupby('image_id')['id'].count()
    ax.hist(opc, bins=40, edgecolor='none', alpha=0.8)
    ax.axvline(opc.mean(), color='red', linestyle='--', label=f'Mean={opc.mean():.1f}')
    ax.axvline(opc.median(), color='green', linestyle='--', label=f'Median={opc.median():.0f}')
    ax.set_title(f'{split_name} — Objects per Image', fontweight='bold')
    ax.set_xlabel('Objects')
    ax.set_ylabel('Images')
    ax.legend(fontsize=8)

plt.suptitle('Object Density Distribution per Split', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Objects per image broken down by clutter level (val + test only)
checkout_ann = pd.DataFrame(checkout_js['annotations'])
checkout_img = pd.DataFrame(checkout_js['images'])[['id', 'level']]
checkout_img.rename(columns={'id': 'image_id'}, inplace=True)
checkout_ann = checkout_ann.merge(checkout_img, on='image_id', how='left')

opc_level = checkout_ann.groupby(['image_id', 'level'])['id'].count().reset_index()
opc_level.columns = ['image_id', 'level', 'n_objects']

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=opc_level, x='level', y='n_objects', order=['easy', 'medium', 'hard'],
            palette={'easy': '#55A868', 'medium': '#DD8452', 'hard': '#C44E52'}, ax=ax)
ax.set_title('Objects per Image by Clutter Level (Checkout)', fontweight='bold')
ax.set_xlabel('Clutter Level')
ax.set_ylabel('Number of Objects')
plt.tight_layout()
plt.show()

print(opc_level.groupby('level')['n_objects'].describe())

---
## 11. Annotated Sample Visualization

In [ ]:
def draw_annotations(img_path, anns, cat_df, max_w=800):
    """Draw bounding boxes and category labels on an image."""
    img = np.array(Image.open(img_path).convert('RGB'))
    h, w = img.shape[:2]

    # Build color map per super-category
    supers = sorted(cat_df['supercategory'].unique())
    palette = plt.cm.get_cmap('tab20', len(supers))
    super2color = {s: (np.array(palette(i)[:3]) * 255).astype(int).tolist()
                   for i, s in enumerate(supers)}
    id2cat = cat_df.set_index('id').to_dict('index')

    fig, ax = plt.subplots(figsize=(min(max_w, w) / 72, h / 72 * (min(max_w, w) / w)))
    ax.imshow(img)
    ax.axis('off')

    for _, ann in anns.iterrows():
        x, y, bw, bh = ann['bbox']
        cat_info = id2cat.get(ann['category_id'], {})
        label    = cat_info.get('name', str(ann['category_id']))
        sup      = cat_info.get('supercategory', '')
        color    = [c/255 for c in super2color.get(sup, [255, 0, 0])]
        rect = patches.Rectangle((x, y), bw, bh,
                                  linewidth=1.5, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x, y - 4, label, fontsize=5, color='white',
                bbox=dict(boxstyle='square,pad=0.1', facecolor=color, alpha=0.8, edgecolor='none'))
    return fig

In [ ]:
# ── Visualise 6 random TRAIN images (single product) ─────────────────────
train_imgs = pd.DataFrame(train_js['images'])
train_anns = pd.DataFrame(train_js['annotations'])
samples    = train_imgs.sample(6, random_state=7)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (_, row) in zip(axes.flat, samples.iterrows()):
    img_path = os.path.join(DATA_ROOT, 'train2019', row['file_name'])
    if not os.path.exists(img_path):
        ax.axis('off')
        ax.set_title('Image not found', color='red')
        continue
    img  = Image.open(img_path).convert('RGB')
    anns = train_anns[train_anns['image_id'] == row['id']]
    ax.imshow(img)
    for _, ann in anns.iterrows():
        x, y, bw, bh = ann['bbox']
        rect = patches.Rectangle((x, y), bw, bh,
                                  linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        ax.text(x, y-4, id2name.get(ann['category_id'], '?'),
                fontsize=7, color='white',
                bbox=dict(boxstyle='square,pad=0.1', facecolor='green', alpha=0.7))
    ax.set_title(row['file_name'][:30], fontsize=8)
    ax.axis('off')

plt.suptitle('Train Images — Single Product (bbox overlay)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualise 1 checkout image per clutter level ──────────────────────────
val_imgs_df  = pd.DataFrame(val_js['images'])
val_anns_df  = pd.DataFrame(val_js['annotations'])

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, level in zip(axes, ['easy', 'medium', 'hard']):
    lvl_imgs = val_imgs_df[val_imgs_df['level'] == level]
    if lvl_imgs.empty:
        ax.axis('off')
        continue
    row      = lvl_imgs.sample(1, random_state=42).iloc[0]
    img_path = os.path.join(DATA_ROOT, 'val2019', row['file_name'])
    if not os.path.exists(img_path):
        ax.axis('off')
        ax.set_title(f'{level} — image not found', color='red')
        continue
    img  = Image.open(img_path).convert('RGB')
    anns = val_anns_df[val_anns_df['image_id'] == row['id']]
    ax.imshow(img)
    for _, ann in anns.iterrows():
        x, y, bw, bh = ann['bbox']
        rect = patches.Rectangle((x, y), bw, bh,
                                  linewidth=1.5, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(x, y-2, id2name.get(ann['category_id'], '?'),
                fontsize=6, color='white',
                bbox=dict(boxstyle='square,pad=0.1', facecolor='red', alpha=0.6))
    ax.set_title(f'Level: {level.upper()}  |  {len(anns)} products', fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('Checkout Scenes — Easy / Medium / Hard Clutter', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 12. Unique Categories per Checkout Image

In [ ]:
cats_per_img = checkout_ann.groupby('image_id')['category_id'].nunique().reset_index()
cats_per_img.columns = ['image_id', 'n_categories']
cats_per_img = cats_per_img.merge(
    checkout_img.rename(columns={'image_id':'image_id'}), on='image_id', how='left')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(cats_per_img['n_categories'], bins=30, edgecolor='none', alpha=0.8)
axes[0].set_title('Unique SKU Categories per Checkout Image', fontweight='bold')
axes[0].set_xlabel('Unique categories')
axes[0].set_ylabel('Images')

sns.boxplot(data=cats_per_img, x='level', y='n_categories',
            order=['easy', 'medium', 'hard'],
            palette={'easy': '#55A868', 'medium': '#DD8452', 'hard': '#C44E52'},
            ax=axes[1])
axes[1].set_title('Unique Categories per Image by Clutter Level', fontweight='bold')
axes[1].set_xlabel('Clutter Level')
axes[1].set_ylabel('Unique SKU Categories')

plt.tight_layout()
plt.show()

print(cats_per_img.groupby('level')['n_categories'].describe())

---
## 13. EDA Summary & Key Insights

| Observation | Implication for YOLOv8x training |
|---|---|
| **200 fine-grained SKU classes** with moderate class imbalance | Use weighted loss or focal loss; consider data augmentation for rare SKUs |
| Train images are **single-product, clean** backgrounds | Strong one-shot reference; augment with copy-paste into checkout scenes |
| Checkout images have **multi-scale, overlapping** objects | Use multi-scale inference; enable NMS tuning |
| Three clutter levels reflect **real-world complexity** | Evaluate separately per level; hard ≈ production environment |
| BBox aspect ratios **cluster near 1:1** | Default YOLO anchors are a good starting point |
| Relative object size is **small–medium** (< 20% of image) | P2/P3 feature maps matter; avoid aggressive downsampling |
| ~47 objects per checkout image on average | High density demands good NMS and anchor coverage |

In [ ]:
print('=' * 60)
print('  RPC DATASET — EDA SUMMARY')
print('=' * 60)
print(f'  Total categories    : {len(cat_df)}')
print(f'  Super-categories    : {cat_df["supercategory"].nunique()}')
print(f'  Train images        : {len(train_js["images"]):,}')
print(f'  Val images          : {len(val_js["images"]):,}')
print(f'  Test images         : {len(test_js["images"]):,}')
print(f'  Total annotations   : {len(all_ann):,}')
print(f'  Clutter levels      : easy / medium / hard')
total_train = all_ann[all_ann['split']=='train']['category_id'].nunique()
print(f'  Categories in train : {total_train}')
print('=' * 60)